In [0]:
# =============================================
# Cell 1. PostgreSQL JDBC 드라이버 확인
# =============================================
try:
    spark._jvm.Class.forName("org.postgresql.Driver")
    print("✅ PostgreSQL 드라이버 확인!")
except:
    print("❌ 드라이버 없음 → Cluster Libraries에서 Maven 설치 필요")
    print("   org.postgresql:postgresql:42.7.3")

✅ PostgreSQL 드라이버 확인!


In [0]:
# =============================================
# Cell 2. 연결 설정
# =============================================
jdbc_url = "jdbcurl"

connection_properties = {
    "user": "dogwalk_admin",
    "password": "<password>",
    "driver": "org.postgresql.Driver",
    "sslmode": "require"
}

In [0]:
# =============================================
# Cell 3. 연결 테스트
# =============================================
test_df = spark.read.jdbc(
    url=jdbc_url,
    table="(SELECT 1 AS test) AS t",
    properties=connection_properties
)
test_df.show()
print("✅ PostgreSQL 연결 성공!")

+----+
|test|
+----+
|   1|
+----+

✅ PostgreSQL 연결 성공!


In [0]:
# =============================================
# Cell 4. Gold → PostgreSQL 적재
# =============================================
df_gold = spark.read.format("delta").table("gold.walk_environment")

print(f"📦 적재할 행 수: {df_gold.count()}")

df_gold.write.jdbc(
    url=jdbc_url,
    table="walk_environment",
    mode="overwrite",
    properties=connection_properties
)

print("✅ walk_environment 적재 완료! 🐾")

📦 적재할 행 수: 22
✅ walk_environment 적재 완료! 🐾


In [0]:
# 적재 확인
verify_df = spark.read.jdbc(
    url=jdbc_url,
    table="walk_environment",
    properties=connection_properties
)
verify_df.show(5)
print(f"PostgreSQL 행 수: {verify_df.count()}")

+-----------+------------------+----+-------------+--------+--------+----------+-------------+-----------+--------------------------------+--------+------------+----+----------+----+----------+-------+----------------------------------+----------------+----------------+----------------------------------+--------------+--------------+----------------+----------------+----------------+-----------------+---------+--------+-----------+-----------------+-----------------+--------------------+--------------------+------------------+
|   district|           AREA_NM|TEMP|SENSIBLE_TEMP|HUMIDITY|WIND_SPD|WIND_DIRCT|PRECIPITATION|PRECPT_TYPE|                         PCP_MSG|UV_INDEX|UV_INDEX_LVL|PM10|PM10_INDEX|PM25|PM25_INDEX|AIR_IDX|                           AIR_MSG|    WEATHER_TIME|AREA_CONGEST_LVL|                  AREA_CONGEST_MSG|AREA_PPLTN_MIN|AREA_PPLTN_MAX|      PPLTN_TIME|ROAD_TRAFFIC_SPD|ROAD_TRAFFIC_IDX|ROAD_TRAFFIC_TIME|ACDNT_CNT|EVENT_YN|DST_MSG_CNT|   sdot_avg_noise|   sdot_max_no

In [0]:
# 공원 테이블 적재
df = spark.read.format("delta").table("silver.park")

df.write.jdbc(
    url=jdbc_url,
    table="park",
    mode="overwrite",
    properties=connection_properties
)


In [0]:
# 공원테이블 검증
park_delta_cnt = spark.table("silver.park").count()

park_mysql_cnt = spark.read.jdbc(
    url=jdbc_url,
    table="park",
    properties=connection_properties
).count()

print(f"- Park Delta Count: {park_delta_cnt}")
print(f"- Park MySQL Count: {park_mysql_cnt}")

In [0]:
# 골드레이어 테이블 적재
df_walk_features = spark.read.format("delta").table("gold.scored")

df_walk_features.write.jdbc(
    url=jdbc_url,
    table="walk_features",
    mode="overwrite",
    properties=connection_properties
)


In [0]:
# 골드레이어 테이블 검증
gold_delta_cnt = spark.table("gold.scored").count()

gold_mysql_cnt = spark.read.jdbc(
    url=jdbc_url,
    table="walk_features",
    properties=connection_properties
).count()

print(f"- Gold Delta Count: {gold_delta_cnt}")
print(f"- Gold MySQL Count: {gold_mysql_cnt}")